# XGBoost Adaptive Resource Allocation Model
### Hybrid DE-WOA Cluster Manager

**Goal:** Predict optimal fitness weights `(w_cpu, w_ram, w_io, w_energy)` and dynamic thresholds  
`(thresh_cpu, thresh_ram, thresh_http)` from live cluster metrics, with **incremental learning**  
so the model adapts every 5 minutes as new data arrives.

---

### Architecture Overview
```
InfluxDB metrics  ──►  Feature Engineering  ──►  7 XGBRegressors  ──►  Predicted weights & thresholds
                                                       │
                                              Incremental Update
                                           (every 5 min, +20 trees)
```

### Why 7 separate models (not one MultiOutput)?
- XGBoost's `xgb_model=` parameter enables **true incremental learning** (adds new trees on top)
- `MultiOutputRegressor` wraps sklearn's interface and breaks the `xgb_model=` warm-start
- Each target has a different scale and noise profile — separate models give better R²
- Each `.ubj` file is tiny (~500KB) and loads in milliseconds in `main.py`

## 1. Imports & Configuration

In [1]:
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# ── Constants matching the simulation ──────────────────────────────────────────
SCRAPE_INTERVAL = 15          # seconds between Prometheus scrapes
MODELS_DIR      = './models'  # where .ubj model files are saved
DATA_DIR        = './data'
os.makedirs(MODELS_DIR, exist_ok=True)

# ── All 7 prediction targets ───────────────────────────────────────────────────
TARGET_WEIGHTS = ['w_cpu', 'w_ram', 'w_io', 'w_energy']
TARGET_THRESH  = ['thresh_cpu', 'thresh_ram', 'thresh_http']
ALL_TARGETS    = TARGET_WEIGHTS + TARGET_THRESH

print(f"XGBoost version : {xgb.__version__}")
print(f"Targets to predict: {ALL_TARGETS}")

ModuleNotFoundError: No module named 'xgboost'

## 2. Load Raw Data

In [ ]:
df_metrics = pd.read_csv(f'{DATA_DIR}/1_influxdb_raw_metrics.csv')
df_logs    = pd.read_csv(f'{DATA_DIR}/2_cluster_manager_logs.csv')

print(f"Metrics shape : {df_metrics.shape}")
print(f"Logs shape    : {df_logs.shape}")
print(f"\nUnique instances : {df_metrics['instance'].nunique()}")
print(f"Time range       : {df_metrics['_time'].min()}  →  {df_metrics['_time'].max()}")

df_metrics.head(3)

In [ ]:
df_logs.head(3)

## 3. Feature Engineering

The raw CSV contains **cumulative counters** (e.g. `node_cpu_seconds_total_idle` keeps growing).  
XGBoost needs **instantaneous rates** — we compute per-instance deltas between consecutive scrapes.

| Raw Column | Derived Feature | Formula |
|---|---|---|
| `node_cpu_seconds_total_idle` | `cpu_busy_pct` | `(1 - Δidle / scrape_interval) * 100` |
| `node_memory_MemAvailable_bytes` | `ram_usage_pct` | `(1 - avail/total) * 100` |
| `node_disk_io_time_seconds_total` | `io_util_pct` | `Δio_time / scrape_interval * 100` |
| `http_requests_total_5xx` / `all` | `http_5xx_rate` | `Δ5xx / Δall` |
| `node_network_receive_drop_total` | `net_drop_rate` | `Δdrops / Δpackets` |
| `scaph_vm_power_microwatts` | `power_watts` | `microwatts / 1e6` |

In [ ]:
# ── Sort so that per-instance diff() is chronologically correct ────────────────
df = df_metrics.sort_values(['instance', '_time']).reset_index(drop=True)

# ── 1. RAM usage % (no delta needed — it's an instant gauge) ──────────────────
df['ram_usage_pct'] = (
    1 - df['node_memory_MemAvailable_bytes'] / df['node_memory_MemTotal_bytes']
) * 100

# ── 2. CPU busy %  (diff of cumulative idle counter per instance) ──────────────
df['_cpu_idle_delta'] = df.groupby('instance')['node_cpu_seconds_total_idle'].diff()
# First scrape per instance has NaN delta → fill with 0 (no change known yet)
df['_cpu_idle_delta'] = df['_cpu_idle_delta'].fillna(0).clip(lower=0)
# idle_rate: fraction of the interval spent idle  (capped to [0,1])
idle_rate = (df['_cpu_idle_delta'] / SCRAPE_INTERVAL).clip(0, 1)
df['cpu_busy_pct'] = (1 - idle_rate) * 100

# ── 3. Disk IO utilisation % ──────────────────────────────────────────────────
df['_io_delta'] = df.groupby('instance')['node_disk_io_time_seconds_total'].diff().fillna(0).clip(lower=0)
df['io_util_pct'] = (df['_io_delta'] / SCRAPE_INTERVAL * 100).clip(0, 100)

# ── 4. HTTP 5xx error rate ────────────────────────────────────────────────────
df['_http_all_delta']  = df.groupby('instance')['http_requests_total_all'].diff().fillna(0).clip(lower=1)
df['_http_5xx_delta']  = df.groupby('instance')['http_requests_total_5xx'].diff().fillna(0).clip(lower=0)
df['http_5xx_rate']    = (df['_http_5xx_delta'] / df['_http_all_delta']).clip(0, 1)

# ── 5. Network packet drop rate ───────────────────────────────────────────────
df['_net_pkt_delta']  = df.groupby('instance')['node_network_receive_packets_total'].diff().fillna(0).clip(lower=1)
df['_net_drop_delta'] = df.groupby('instance')['node_network_receive_drop_total'].diff().fillna(0).clip(lower=0)
df['net_drop_rate']   = (df['_net_drop_delta'] / df['_net_pkt_delta']).clip(0, 1)

# ── 6. Power in watts (human-readable scale) ─────────────────────────────────
df['power_watts'] = df['scaph_vm_power_microwatts'] / 1_000_000

# ── 7. Role one-hot flags (derived from vm_name) ─────────────────────────────
df['is_worker']  = df['vm_name'].str.contains('worker',              case=False).fillna(False).astype(int)
df['is_master']  = df['vm_name'].str.contains('master',              case=False).fillna(False).astype(int)
df['is_monitor'] = df['vm_name'].str.contains('monitoring|influx|snmp', case=False, regex=True).fillna(False).astype(int)

# ── 8. VLAN ordinal encoding ──────────────────────────────────────────────────
vlan_encoder = LabelEncoder()
df['vlan_enc'] = vlan_encoder.fit_transform(df['vlan'])
print("VLAN encoding:", dict(zip(vlan_encoder.classes_, vlan_encoder.transform(vlan_encoder.classes_))))

# ── 9. Incident flags (from influxdb columns) ─────────────────────────────────
# scrape_duration_seconds > 2 → High_Latency; up == 0 → VM_Unreachable
# These are already present in the metrics CSV and carry rich predictive signal

# ── Drop intermediate helper columns ──────────────────────────────────────────
helper_cols = ['_cpu_idle_delta', '_io_delta', '_http_all_delta', '_http_5xx_delta', '_net_pkt_delta', '_net_drop_delta']
df.drop(columns=helper_cols, inplace=True)

print(f"\nFeature engineering complete. Shape: {df.shape}")
df[['instance','cpu_busy_pct','ram_usage_pct','io_util_pct','http_5xx_rate','net_drop_rate','power_watts']].describe()

## 4. Merge Metrics with Manager Logs (targets)

In [ ]:
# Only keep rows that appear in both datasets (inner join on time+instance+vm_name)
# This drops monitoring-only rows and the first scrape per instance (no delta yet)
df_merged = pd.merge(
    df, df_logs,
    on=['_time', 'instance', 'vm_name'],
    how='inner'
)

print(f"Merged shape: {df_merged.shape}")
print(f"Columns     : {df_merged.columns.tolist()}")

## 5. Define X (Features) and Y (Targets)

### What goes in X and why

| Feature | Why it matters |
|---|---|
| `up` | VM reachability — unreachable nodes should get zero weight |
| `scrape_duration_seconds` | High latency → CPU bottleneck indicator |
| `cpu_busy_pct` | Primary driver of `w_cpu` and `thresh_cpu` |
| `ram_usage_pct` | Primary driver of `w_ram` and `thresh_ram` |
| `io_util_pct` | Primary driver of `w_io` |
| `http_5xx_rate` | Error rate — drives `thresh_http` |
| `net_drop_rate` | Network congestion signal |
| `power_watts` | Primary driver of `w_energy` |
| `is_worker / is_master / is_monitor` | Role context — workers tolerate higher CPU |
| `vlan_enc` | Network zone — affects latency and packet policy |

### What does NOT go in X
- `_time` — timestamp (not a feature, causes data leakage)
- `instance` — IP string (identifier, not a metric)
- `vm_name` — string identifier
- `pm_ip` — physical host IP (string identifier)
- Raw cumulative counters — replaced by derived rate features above
- `incident_type` from logs — this is a result of conditions, not a predictor of weights

In [ ]:
FEATURE_COLS = [
    'up',
    'scrape_duration_seconds',
    'cpu_busy_pct',
    'ram_usage_pct',
    'io_util_pct',
    'http_5xx_rate',
    'net_drop_rate',
    'power_watts',
    'is_worker',
    'is_master',
    'is_monitor',
    'vlan_enc',
]

# Keep only rows where all features AND all targets are non-null
df_model = df_merged[FEATURE_COLS + ALL_TARGETS].dropna()

X = df_model[FEATURE_COLS]
y = df_model[ALL_TARGETS]

print(f"Final dataset: {X.shape[0]} samples × {X.shape[1]} features")
print(f"Targets      : {ALL_TARGETS}")
print(f"\nFeature stats:")
X.describe().round(3)

In [ ]:
# Quick look at target distributions
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

colors = ['#4C72B0','#55A868','#C44E52','#8172B2','#CCB974','#64B5CD','#E07B54']
for i, (target, color) in enumerate(zip(ALL_TARGETS, colors)):
    axes[i].hist(y[target], bins=40, color=color, alpha=0.75, edgecolor='white')
    axes[i].set_title(target, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
    axes[i].grid(axis='y', alpha=0.3)

axes[-1].set_visible(False)  # hide extra subplot
plt.suptitle('Target Variable Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('./models/target_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Train/Test Split

> ⚠️ **Important**: `stratify=` is **only for classification** (it balances class labels).  
> For multi-output **regression**, use a plain `shuffle=True` split.

In [ ]:
# 80/20 split — shuffle=False pour respecter l'ordre chronologique
# données de monitoring = série temporelle :
# mélanger causerait une fuite du futur vers le passé
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=False,   # ← CORRECTION BUG 2 : False obligatoire pour séries temporelles
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test : {X_test.shape[0]} samples")


## 7. Train One XGBRegressor per Target

Each model is saved as `models/<target>.ubj` — the native XGBoost binary format.  
These files are loaded by `main.py` at startup and reloaded after each incremental update.

In [ ]:
XGB_PARAMS = dict(
    n_estimators    = 300,
    max_depth       = 6,
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    min_child_weight= 3,
    tree_method     = 'hist',         # fast histogram-based splits
    objective       = 'reg:squarederror',
    random_state    = 42,
    early_stopping_rounds = 30,       # stop if no improvement for 30 rounds
    eval_metric     = 'mae',
)

models = {}

for target in ALL_TARGETS:
    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(
        X_train, y_train[target],
        eval_set=[(X_test, y_test[target])],
        verbose=False,
    )
    models[target] = model
    model.save_model(f'{MODELS_DIR}/{target}.ubj')
    best = model.best_iteration
    print(f"  [{target:15s}]  best_iteration={best}  "
          f"MAE_train={mean_absolute_error(y_train[target], model.predict(X_train)):.4f}  "
          f"MAE_test={mean_absolute_error(y_test[target], model.predict(X_test)):.4f}")

print(f"\n✓  7 models saved to '{MODELS_DIR}/'")

## 8. Evaluation

In [ ]:
print(f"{'Target':15s}  {'MAE':>8s}  {'R²':>7s}")
print("-" * 35)
results = {}
for target in ALL_TARGETS:
    preds = models[target].predict(X_test)
    mae   = mean_absolute_error(y_test[target], preds)
    r2    = r2_score(y_test[target], preds)
    results[target] = {'mae': mae, 'r2': r2}
    quality = '✓' if r2 > 0.85 else ('~' if r2 > 0.6 else '✗')
    print(f"{target:15s}  {mae:8.4f}  {r2:7.4f}  {quality}")

In [ ]:
# Feature importances for the 4 weight models
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, target in enumerate(TARGET_WEIGHTS):
    importances = pd.Series(
        models[target].feature_importances_,
        index=FEATURE_COLS
    ).sort_values(ascending=True)

    importances.plot(kind='barh', ax=axes[i], color='#4C72B0', alpha=0.8)
    axes[i].set_title(f'Feature Importance — {target}', fontweight='bold')
    axes[i].set_xlabel('Importance Score')
    axes[i].grid(axis='x', alpha=0.3)

plt.suptitle('XGBoost Feature Importances (Fitness Weights)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./models/feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Incremental Learning

XGBoost supports **warm-start incremental training** via the `xgb_model=` parameter of `xgb.train()`.  
This **appends new trees** on top of the existing booster — old knowledge is preserved.

```
Existing model (300 trees)  +  New 5-min batch  →  Updated model (320 trees)
```

This function is called by `main.py` every 5 minutes after collecting fresh metrics.

In [ ]:
def engineer_features(df_raw: pd.DataFrame, vlan_enc: LabelEncoder) -> pd.DataFrame:
    """
    Re-usable feature engineering pipeline.
    Input  : raw metrics DataFrame (same schema as 1_influxdb_raw_metrics.csv)
    Output : DataFrame with FEATURE_COLS ready for XGBoost
    """
    df = df_raw.sort_values(['instance', '_time']).reset_index(drop=True).copy()

    df['ram_usage_pct']  = (1 - df['node_memory_MemAvailable_bytes'] / df['node_memory_MemTotal_bytes']) * 100

    cpu_idle_d           = df.groupby('instance')['node_cpu_seconds_total_idle'].diff().fillna(0).clip(lower=0)
    df['cpu_busy_pct']   = (1 - (cpu_idle_d / SCRAPE_INTERVAL).clip(0, 1)) * 100

    io_d                 = df.groupby('instance')['node_disk_io_time_seconds_total'].diff().fillna(0).clip(lower=0)
    df['io_util_pct']    = (io_d / SCRAPE_INTERVAL * 100).clip(0, 100)

    http_all_d           = df.groupby('instance')['http_requests_total_all'].diff().fillna(0).clip(lower=1)
    http_5xx_d           = df.groupby('instance')['http_requests_total_5xx'].diff().fillna(0).clip(lower=0)
    df['http_5xx_rate']  = (http_5xx_d / http_all_d).clip(0, 1)

    pkt_d                = df.groupby('instance')['node_network_receive_packets_total'].diff().fillna(0).clip(lower=1)
    drop_d               = df.groupby('instance')['node_network_receive_drop_total'].diff().fillna(0).clip(lower=0)
    df['net_drop_rate']  = (drop_d / pkt_d).clip(0, 1)

    df['power_watts']    = df['scaph_vm_power_microwatts'] / 1_000_000
    df['is_worker']      = df['vm_name'].str.contains('worker', case=False).astype(int)
    df['is_master']      = df['vm_name'].str.contains('master', case=False).astype(int)
    df['is_monitor']     = df['vm_name'].str.contains('monitoring|influx|snmp', case=False, regex=True).astype(int)

    # Handle unseen VLAN labels gracefully
    known_vlans = set(vlan_enc.classes_)
    df['vlan_safe'] = df['vlan'].where(df['vlan'].isin(known_vlans), other=vlan_enc.classes_[0])
    df['vlan_enc']  = vlan_enc.transform(df['vlan_safe'])

    return df

print("engineer_features() defined ✓")

In [ ]:
def incremental_update(
    new_metrics_csv: str,
    new_logs_csv: str,
    n_new_trees: int = 20,
    models_dir: str = MODELS_DIR,
) -> dict:
    """
    [UPDATED] Incremental update placeholder.
    Prevents model collapse by disabling soft-label self-training.
    Awaits ground-truth telemetry implementation.
    """
    # 1. Load and engineer new batch
    df_m_raw = pd.read_csv(new_metrics_csv)
    df_l_raw = pd.read_csv(new_logs_csv)

    df_m = engineer_features(df_m_raw, vlan_encoder)
    df_new = pd.merge(df_m, df_l_raw, on=['_time', 'instance', 'vm_name'], how='inner')
    df_new = df_new[FEATURE_COLS + ALL_TARGETS].dropna()

    if len(df_new) < 10:
        print("⚠️ Not enough new samples for update (need ≥10).")
        return {}

    X_new = df_new[FEATURE_COLS]

    update_report = {}

    for target in ALL_TARGETS:
        # 2. Load existing booster from disk
        existing_booster = xgb.Booster()
        existing_booster.load_model(f'{models_dir}/{target}.ubj')

        # --- BUG-13 FIX: MODEL COLLAPSE PREVENTION ---
        # Training a model on its own predictions (soft labels) without ground truth 
        # causes variance collapse. We bypass training until real labels are provided.
        print(f"[{target:15s}] ⚠️ Skipping soft-label training. Awaiting ground-truth pipeline.")
        
        update_report[target] = {
            'mae_before': 0.0,
            'mae_after' : 0.0,
            'delta'     : 0.0,
            'status'    : 'bypassed_no_ground_truth'
        }

    print(f"\n✓ Incremental update bypassed safely.")
    return update_report

print("incremental_update() defined ✓")

In [ ]:
demo_metrics_path = f'{DATA_DIR}/demo_new_metrics.csv'
demo_logs_path    = f'{DATA_DIR}/demo_new_logs.csv'

demo_rows = df_merged.loc[X_test.index].copy()
demo_rows[df_metrics.columns.tolist()].to_csv(demo_metrics_path, index=False)
demo_rows[df_logs.columns.tolist()].to_csv(demo_logs_path, index=False)

print("Running incremental update demo on test batch...")
report = incremental_update(demo_metrics_path, demo_logs_path, n_new_trees=20)

## 10. Prediction Function

This function is imported directly in `main.py` to serve the `/predict-config` endpoint.

In [ ]:
def load_models(models_dir: str = MODELS_DIR) -> dict[str, xgb.Booster]:
    """
    Load all 7 trained boosters from disk.
    Called once at startup in main.py lifespan().
    """
    loaded = {}
    for target in ALL_TARGETS:
        b = xgb.Booster()
        b.load_model(f'{models_dir}/{target}.ubj')
        loaded[target] = b
    print(f"Loaded {len(loaded)} models from '{models_dir}'")
    return loaded


def predict_config(current_metrics: dict, boosters: dict) -> dict:
    """
    Predict optimal weights and thresholds for one VM snapshot.

    Parameters
    ----------
    current_metrics : dict with keys matching FEATURE_COLS
                      (e.g. from Prometheus instant query)
    boosters        : dict[target_name → xgb.Booster], from load_models()

    Returns
    -------
    dict with predicted and post-processed weights + thresholds
    """
    X_input = xgb.DMatrix(
        pd.DataFrame([current_metrics])[FEATURE_COLS],
        feature_names=FEATURE_COLS
    )

    raw = {target: float(boosters[target].predict(X_input)[0]) for target in ALL_TARGETS}

    # ── Post-processing 1: Normalise weights to sum exactly to 1.0 ────────────
    w_sum = sum(max(0.0, raw[t]) for t in TARGET_WEIGHTS)
    if w_sum > 0:
        for t in TARGET_WEIGHTS:
            raw[t] = max(0.0, raw[t]) / w_sum
    else:
        # Fallback to equal weights if model outputs negatives
        for t in TARGET_WEIGHTS:
            raw[t] = 0.25

    # ── Post-processing 2: Clamp thresholds to safe operating ranges ──────────
    raw['thresh_cpu']  = float(np.clip(raw['thresh_cpu'],  50.0, 95.0))
    raw['thresh_ram']  = float(np.clip(raw['thresh_ram'],  55.0, 95.0))
    raw['thresh_http'] = float(np.clip(raw['thresh_http'],  0.1,  5.0))

    return raw


# ── Smoke test ─────────────────────────────────────────────────────────────────
boosters = load_models()

sample_input = {
    'up'                      : 1.0,
    'scrape_duration_seconds' : 0.12,
    'cpu_busy_pct'            : 78.5,
    'ram_usage_pct'           : 65.3,
    'io_util_pct'             : 12.0,
    'http_5xx_rate'           : 0.03,
    'net_drop_rate'           : 0.001,
    'power_watts'             : 95.0,
    'is_worker'               : 1,
    'is_master'               : 0,
    'is_monitor'              : 0,
    'vlan_enc'                : 1,
}

result = predict_config(sample_input, boosters)
print("\nSample prediction:")
for k, v in result.items():
    print(f"  {k:15s} = {v:.4f}")
print(f"\n  Weight sum check: {sum(result[t] for t in TARGET_WEIGHTS):.6f}  (should be 1.0)")

## 11. Save Artefacts for main.py

In [ ]:
import json

# Save VLAN encoder classes so main.py can rebuild it
joblib.dump(vlan_encoder, f'{MODELS_DIR}/vlan_encoder.joblib')

# Save feature column order (critical — must match main.py exactly)
with open(f'{MODELS_DIR}/feature_cols.json', 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)

# Save evaluation results
pd.DataFrame(results).T.to_csv(f'{MODELS_DIR}/evaluation_results.csv')

print("Saved:")
for fname in os.listdir(MODELS_DIR):
    fpath = os.path.join(MODELS_DIR, fname)
    print(f"  {fname:40s}  {os.path.getsize(fpath) / 1024:.1f} KB")

---
## 13. Intégration avec ml_service.py

Ce bloc connecte le notebook à l'API FastAPI (`ml_service.py`).

```
NOTEBOOK (Google Colab)
  └── Entraîne les 7 modèles → sauvegarde les .ubj
                                      │
                          (copier le dossier models/)
                                      │
                                      ▼
ML SERVICE (uvicorn ml_service:app --port 8001)
  └── Charge les .ubj au démarrage
  └── POST /sync  ← reçoit les snapshots VM de main.py
                   → incremental update toutes les 5 min
                   → retourne weights + thresholds
```

Les cellules ci-dessous permettent de :
1. Vérifier que le service tourne et que les modèles sont chargés
2. Tester le endpoint `/sync` depuis le notebook
3. Déclencher un incremental update manuel via l'API


In [ ]:
# ── Vérifier que ml_service tourne et que les modèles sont chargés ────────────
import requests

ML_SERVICE_URL = "http://localhost:8001"   # ← adapter si le service est ailleurs

try:
    r = requests.get(f"{ML_SERVICE_URL}/health", timeout=5)
    health = r.json()
    print("=== /health ===")
    print(f"  Status         : {health['status']}")
    print(f"  Modèles chargés: {health['models_loaded']} / {len(ALL_TARGETS)}")
    print(f"  Prêt           : {health['models_ready']}")
    print(f"  Dernier update : {health.get('last_update', 'jamais')}")

    r2 = requests.get(f"{ML_SERVICE_URL}/model-info", timeout=5)
    info = r2.json()
    print("\n=== /model-info ===")
    print(f"  Targets      : {info['targets']}")
    print(f"  Features     : {info['feature_cols']}")
    print(f"  Arbres/modèle: {info['tree_counts']}")

except requests.exceptions.ConnectionError:
    print("❌ ml_service n'est pas démarré.")
    print("   Lance : uvicorn ml_service:app --host 0.0.0.0 --port 8001")


In [ ]:
# ── Tester POST /sync avec des snapshots construits depuis les données de test ─
#
# En production c'est main.py qui envoie les snapshots toutes les 5 minutes.
# Ici on simule ce comportement depuis le notebook avec les données de test.

def build_snapshots_from_test(df_src: pd.DataFrame, n: int = 50) -> list:
    """
    Convertit des lignes du DataFrame test en format VMSnapshot
    attendu par POST /sync de ml_service.

    Paramètres
    ----------
    df_src : DataFrame avec les colonnes FEATURE_COLS
    n      : nombre de snapshots à envoyer (simule n VMs)
    """
    # Récupérer les lignes test originales (avec vm_name, vlan, etc.)
    src_rows = df_merged.loc[X_test.index].head(n).copy()

    snapshots = []
    for _, row in src_rows.iterrows():
        snapshots.append({
            "instance"        : str(row.get("instance",        "10.0.0.1:9100")),
            "vm_name"         : str(row.get("vm_name",         "k8s-worker-1")),
            "vlan"            : str(row.get("vlan",            "vlan-1-app")),
            "up"              : float(row.get("up",             1.0)),
            "scrape_duration" : float(row.get("scrape_duration_seconds", 0.1)),
            "cpu_pct"         : float(row.get("cpu_busy_pct",  50.0)),
            "ram_pct"         : float(row.get("ram_usage_pct", 60.0)),
            "io_pct"          : float(row.get("io_util_pct",   5.0)),
            "http_5xx_rate"   : float(row.get("http_5xx_rate", 0.01)),
            "net_drop_rate"   : float(row.get("net_drop_rate", 0.001)),
            "power_watts"     : float(row.get("power_watts",   80.0)),
        })
    return snapshots


try:
    snapshots = build_snapshots_from_test(df_merged, n=100)

    payload = {
        "snapshots": snapshots,
        "sent_at"  : pd.Timestamp.now(tz="UTC").isoformat()
    }

    print(f"Envoi de {len(snapshots)} snapshots à POST /sync ...")
    r = requests.post(
        f"{ML_SERVICE_URL}/sync",
        json    = payload,
        timeout = 60
    )

    if r.status_code == 200:
        resp = r.json()
        print(f"\n✅ /sync OK — {resp['n_samples']} samples traités")
        print("\n=== Config retournée ===")
        cfg = resp['config']
        print(f"  Weights  : cpu={cfg['w_cpu']:.3f}  ram={cfg['w_ram']:.3f}  io={cfg['w_io']:.3f}  energy={cfg['w_energy']:.3f}")
        print(f"  CPU      : warn={cfg['thresh_cpu_warn']}  crit={cfg['thresh_cpu_crit']}")
        print(f"  RAM      : warn={cfg['thresh_ram_warn']}  crit={cfg['thresh_ram_crit']}")
        print(f"  HTTP 5xx : warn={cfg['thresh_http_warn']}  crit={cfg['thresh_http_crit']}")
        print("\n=== Rapport incremental update ===")
        for target, rep in resp['update_report'].items():
            arrow = '↓' if rep['mae_after'] < rep['mae_before'] else ('↑ rollback' if not rep.get('saved', True) else '→')
            print(f"  [{target:20s}] MAE {rep['mae_before']:.5f} → {rep['mae_after']:.5f}  {arrow}  (total arbres: {rep['trees_total']})")
    else:
        print(f"❌ Erreur {r.status_code}: {r.text}")

except requests.exceptions.ConnectionError:
    print("❌ ml_service n'est pas démarré — lance d'abord le service")


In [ ]:
# ── Pipeline complet : Notebook → ml_service → Résultats ──────────────────────
#
# Ce résumé montre comment les deux parties s'articulent.
# À copier dans ton README ou ta documentation de thèse.

PIPELINE = """
╔══════════════════════════════════════════════════════════════════╗
║              PIPELINE COMPLET DU SYSTÈME ML                     ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  PHASE 1 — ENTRAÎNEMENT INITIAL (Google Colab / ce notebook)    ║
║  ─────────────────────────────────────────────────────────────   ║
║  1. Charger données InfluxDB (1_influxdb_raw_metrics.csv)        ║
║  2. Feature engineering (cpu_busy_pct, ram_usage_pct, ...)       ║
║  3. Merge avec logs cluster (2_cluster_manager_logs.csv)         ║
║  4. Entraîner 7 XGBRegressors (w_cpu, w_ram, w_io, w_energy,    ║
║     thresh_cpu, thresh_ram, thresh_http)                         ║
║  5. Sauvegarder → models/*.ubj + vlan_encoder.joblib            ║
║                                                                  ║
║  PHASE 2 — SERVICE EN PRODUCTION (ml_service.py)                ║
║  ─────────────────────────────────────────────────────────────   ║
║  uvicorn ml_service:app --host 0.0.0.0 --port 8001              ║
║                                                                  ║
║  Démarrage → charge les 7 modèles .ubj depuis models/           ║
║                                                                  ║
║  Toutes les 5 min → main.py envoie POST /sync                   ║
║    payload : {snapshots: [{cpu_pct, ram_pct, ...}×N VMs]}       ║
║    réponse : {weights, thresholds, update_report}               ║
║                                                                  ║
║  À chaque /sync :                                                ║
║    a) Prédire weights + thresholds actuels                       ║
║    b) Incremental update (+20 arbres par modèle)                 ║
║    c) Rollback si MAE empire de plus de 5%                      ║
║    d) Sauvegarder les .ubj mis à jour                            ║
║                                                                  ║
║  GET /config  → dernière config sans update                     ║
║  GET /health  → état du service + timestamp dernier update      ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(PIPELINE)

# Vérification finale : les fichiers modèles sont-ils bien présents ?
import os
print("=== Fichiers dans models/ ===")
if os.path.exists(MODELS_DIR):
    for f in sorted(os.listdir(MODELS_DIR)):
        path = os.path.join(MODELS_DIR, f)
        size = os.path.getsize(path) / 1024
        ok   = '✅' if f.endswith('.ubj') or f.endswith('.joblib') or f.endswith('.json') else '📄'
        print(f"  {ok} {f:40s} {size:.1f} KB")
else:
    print("  ❌ Dossier models/ introuvable — lance d'abord les cellules d'entraînement")
